## Model summary

Common notation: $r_t$ = log-return (in %), $\sigma_t^2$ = conditional variance, $x_t$ = exogenous variable (tone or art_growth), $z_t \sim t^*_\nu$ (standardized Student-t, $\nu > 2$), with $r_t = \sigma_t z_t$.

**1. GARCH(1,1) — Student-t baseline**
$$\sigma_t^2 = \omega + \alpha\, r_{t-1}^2 + \beta\, \sigma_{t-1}^2$$

**2. GARCH-X (tone)** — G&P Eq. (4), $x_{t-1} = \text{tone}_{t-1}$
$$\sigma_t^2 = \omega + \alpha\, r_{t-1}^2 + \beta\, \sigma_{t-1}^2 + \gamma\, x_{t-1}^2$$

**3. GARCH-X (article growth)** — G&P Eq. (4), $x_{t-1} = \text{art\_growth}_{t-1}$
$$\sigma_t^2 = \omega + \alpha\, r_{t-1}^2 + \beta\, \sigma_{t-1}^2 + \gamma\, x_{t-1}^2$$

**4. GARCHAND (tone)** — G&P Eq. (5), asymmetry driven by the sign of tone
$$\sigma_t^2 = \omega + \alpha\, r_{t-1}^2 + \beta\, \sigma_{t-1}^2 + \gamma\,(1 + d_1\,\theta)\, x_{t-1}^2,\qquad d_1 = \mathbb{1}\{x_{t-1} < 0\},\ \theta > 0$$

**5. GARCHAND (article growth)** — G&P Eq. (6), news effect active only when volume grows
$$\sigma_t^2 = \omega + \alpha\, r_{t-1}^2 + \beta\, \sigma_{t-1}^2 + \gamma\, d_2\, x_{t-1}^2,\qquad d_2 = \mathbb{1}\{x_{t-1} > 0\},\ \gamma > 0$$

**6. GARCHND (tone / article growth)** — G&P Eq. (7), news effect active only in a high-volatility regime
$$\sigma_t^2 = \omega + \alpha\, r_{t-1}^2 + \beta\, \sigma_{t-1}^2 + \gamma\, d_3\, x_{t-1}^2,\qquad d_3 = \mathbb{1}\{\sigma_{t-1}^2 \geq \kappa\}$$
where $\kappa$ is calibrated from an annualized volatility threshold ($30\%$ and $50\%$): $\kappa = (v_{\text{ann}}/\sqrt{252})^2$. In the implementation $d_3$ is approximated by a smooth sigmoid $1/(1+e^{-K(\sigma_{t-1}^2 - \kappa)})$ with $K = 20$ so the log-likelihood is differentiable.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.set_option('display.float_format', '{:.6f}'.format)
%matplotlib inline

In [2]:
IN_PATH = 'CLEANED DATA/REMX_prices_sentiment_combined.xlsx'

df = pd.read_excel(IN_PATH, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
df = df.set_index('Date')
df['r'] = np.log(df['Close'] / df['Close'].shift(1))
df['r'] = df['r'] - df['r'].mean()  # zero-mean (Bug 4)
r = df['r'].dropna()
r2 = (r ** 2)

In [3]:
# Bug-7 diagnostic: data-density check
# 02-EDA.ipynb does not contain these article-density diagnostics,
# so we add them here. Read-only — no model changes.

_ac  = df['article_count'] if 'article_count' in df.columns else pd.Series(dtype=float)
_tm  = df['tone_mean_x100_winsor'] if 'tone_mean_x100_winsor' in df.columns else df['tone_mean']
_ag  = df['art_growth_winsor'] if 'art_growth_winsor' in df.columns else df['art_growth']
_r2  = df['r'].dropna() ** 2  # crude RV proxy (squared returns)

if len(_ac) > 0:
    print('=== Article-count distribution ===')
    print(_ac.describe(percentiles=[.01,.05,.25,.50,.75,.95,.99]).to_string())
    zero_art_frac = float((_ac == 0).mean())
    print(f'Share of trading days with 0 articles: {zero_art_frac:.3%}')

    # tone_mean fill value for zero-article days is 0 (see 01-data-cleaning)
    _fill_val = 0.0
    zero_tone_frac = float((df['tone_mean'] == _fill_val).mean())
    print(f'Share of trading days where tone_mean == fill value ({_fill_val}): {zero_tone_frac:.3%}')

print()
print('=== tone_mean_x100 and art_growth: raw vs winsorized ===')
for _col in ['tone_mean', 'tone_mean_x100', 'tone_mean_x100_winsor',
             'art_growth', 'art_growth_winsor']:
    if _col in df.columns:
        s_ = df[_col]
        print(f'{_col:30s}: mean={s_.mean():+.4f}  std={s_.std():.4f}  '
              f'min={s_.min():.4f}  max={s_.max():.4f}')

print()
print('=== Correlations with squared returns (crude RV proxy) ===')
print(f'Lag     tone    |tone|   art_growth')
for _lag in [1, 5, 22]:
    _rv = _r2.shift(-_lag)  # forward shift so we see r^2_{t+lag} ~ x_t
    _df_tmp = pd.DataFrame({'tone': _tm, 'abs_tone': _tm.abs(),
                            'ag': _ag, 'rv': _rv}).dropna()
    ct = _df_tmp[['tone','rv']].corr().iloc[0,1]
    ca = _df_tmp[['abs_tone','rv']].corr().iloc[0,1]
    cg = _df_tmp[['ag','rv']].corr().iloc[0,1]
    print(f'{_lag:>3d}   {ct:+.4f}  {ca:+.4f}   {cg:+.4f}')

=== Article-count distribution ===
count   2766.000000
mean     100.419740
std       95.324085
min        0.000000
1%        14.650000
5%        30.000000
25%       53.000000
50%       78.000000
75%      118.000000
95%      239.250000
99%      427.100000
max     2053.000000
Share of trading days with 0 articles: 0.398%
Share of trading days where tone_mean == fill value (0.0): 0.398%

=== tone_mean_x100 and art_growth: raw vs winsorized ===
tone_mean                     : mean=-0.1198  std=1.0939  min=-7.1825  max=3.4485
tone_mean_x100                : mean=-11.9836  std=109.3931  min=-718.2494  max=344.8546
tone_mean_x100_winsor         : mean=-11.5749  std=105.6102  min=-341.7227  max=184.2196
art_growth                    : mean=-0.0800  std=1.2684  min=-1.0000  max=61.0000
art_growth_winsor             : mean=-0.1086  std=0.4616  min=-0.7921  max=1.6472

=== Correlations with squared returns (crude RV proxy) ===
Lag     tone    |tone|   art_growth
  1   +0.0215  -0.0145   +0.0209
 

## 1. GARCH(1,1)

We estimate the canonical GARCH(1,1) on the REMX daily log returns (in percent) with constant mean and Gaussian innovations:

$$r_t = \mu + \varepsilon_t, \qquad \varepsilon_t = \sigma_t z_t, \qquad \sigma_t^2 = \omega + \alpha\,\varepsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2, \qquad z_t \sim \mathcal{N}(0,1).$$

The likelihood is maximized numerically. We report, per parameter, the estimate, standard error, **t-ratio** ($\hat\theta/\widehat{\mathrm{SE}}$) and the two-sided p-value, together with the model-level **log-likelihood, AIC, BIC**. A parameter is judged significant at the 5% level when $|t| > 1.96$ (equivalently $p < 0.05$). Standard errors are robust (sandwich form), so the conclusions also hold as a Quasi-MLE under conditional non-normality.


In [4]:
%%capture cap_garch
from arch import arch_model
from scipy.stats import norm

# 1. Returns in percent (standard scaling for GARCH stability)
r_pct = (df['r'].dropna() * 100.0)

# 2. Specify and fit GARCH(1,1) with constant mean and Gaussian innovations
am = arch_model(r_pct, mean='Zero', vol='GARCH', p=1, q=1, dist='normal')
res = am.fit(disp='off', cov_type='robust')

print(res.summary())
print()

# 3. Coefficient-level diagnostics (estimate / std err / t-ratio / p-value)
CRIT_10PCT = norm.ppf(1 - 0.10 / 2)  # 1.645
CRIT_5PCT  = norm.ppf(1 - 0.05 / 2)  # 1.960
CRIT_1PCT  = norm.ppf(1 - 0.01 / 2)  # 2.576

tbl = pd.DataFrame({
    'estimate':   res.params,
    'std_err':    res.std_err,
    't_ratio':    res.tvalues,
    'p_value':    res.pvalues,
})
tbl['significant_10pct'] = tbl['t_ratio'].abs() > CRIT_10PCT
tbl['significant_5pct']  = tbl['t_ratio'].abs() > CRIT_5PCT
tbl['significant_1pct']  = tbl['t_ratio'].abs() > CRIT_1PCT

print('Per-parameter significance (|t| > 1.645 -> 10%, |t| > 1.96 -> 5%, |t| > 2.576 -> 1%):')
print(tbl.to_string(float_format=lambda x: f'{x: .4f}'))
print()

# 4. Model-level criteria
print(f'Log-likelihood : {res.loglikelihood: .4f}')
print(f'AIC            : {res.aic: .4f}')
print(f'BIC            : {res.bic: .4f}')
print()

# 5. Stationarity / persistence check on the variance equation
alpha_hat = res.params.get('alpha[1]', float('nan'))
beta_hat  = res.params.get('beta[1]',  float('nan'))
persist   = alpha_hat + beta_hat
print(f'alpha[1] + beta[1] = {persist:.4f}   '
      f'({"stationary (<1)" if persist < 1 else "non-stationary (>=1)"})')
print()

# 6. Plain-language verdict at the 1%, 5% and 10% levels
print('Verdict:')
for name, row in tbl.iterrows():
    if row['significant_1pct']:
        tag = 'SIGNIFICANT at 1% (and 5%, 10%)'
    elif row['significant_5pct']:
        tag = 'SIGNIFICANT at 5% (and 10%) only'
    elif row['significant_10pct']:
        tag = 'SIGNIFICANT at 10% only'
    else:
        tag = 'not significant'
    print(f'  {name:<10}  t = {row["t_ratio"]:+.2f}   p = {row["p_value"]:.4f}   -> {tag}')

# 2. GARCH-X (tone) - tone matters?

In [5]:
%%capture cap_garchx_tone

# --- Shared MLE machinery for the five G&P (2019) GARCH variants ------------
# Estimation by Maximum Likelihood with Gaussian innovations z_t ~ N(0,1) and
# zero-mean returns r_t = sigma_t z_t  (G&P Eq. 3).  Robust sandwich standard
# errors V_robust = H^{-1} J H^{-1}, with J the BHHH outer product of
# finite-difference scores (Bollerslev & Wooldridge, 1992; G&P Eqs. 11-12).
# Returns and exogenous variables are scaled to percent to keep the optimizer
# well-conditioned (cf. G&P p. 8).
from scipy.optimize import minimize
from scipy.stats import norm
from statsmodels.tools.numdiff import approx_hess

CRIT_10 = norm.ppf(1 - 0.10 / 2)   # 1.6449
CRIT_5  = norm.ppf(1 - 0.05 / 2)   # 1.9600
CRIT_1  = norm.ppf(1 - 0.01 / 2)   # 2.5758

def _flag(t):
    a = abs(t)
    if a > CRIT_1:  return 'SIGNIFICANT at 1% (and 5%, 10%)'
    if a > CRIT_5:  return 'SIGNIFICANT at 5% (and 10%) only'
    if a > CRIT_10: return 'SIGNIFICANT at 10% only'
    return 'not significant'

def _sandwich_se(theta, neg_ll_fn, per_obs_fn, args):
    H = approx_hess(theta, neg_ll_fn, args=args)
    V_hess = np.linalg.inv(H)
    eps_fd = 1e-5
    T_obs  = len(args[0])
    G = np.empty((T_obs, len(theta)))
    for i in range(len(theta)):
        up = theta.copy(); up[i] += eps_fd
        dn = theta.copy(); dn[i] -= eps_fd
        ll_up, _ = per_obs_fn(up, *args)
        ll_dn, _ = per_obs_fn(dn, *args)
        G[:, i] = (ll_up - ll_dn) / (2 * eps_fd)
    J = G.T @ G
    V = V_hess @ J @ V_hess
    return np.sqrt(np.diag(V))

def _report(param_names, theta, se, ll, T_obs, persist_alpha_beta=None,
            extra_notes=None):
    t_ratios = theta / se
    p_values = 2.0 * (1.0 - norm.cdf(np.abs(t_ratios)))
    tbl = pd.DataFrame({
        'estimate': theta, 'std_err': se,
        't_ratio': t_ratios, 'p_value': p_values,
    }, index=param_names)
    tbl['significant_10pct'] = tbl['t_ratio'].abs() > CRIT_10
    tbl['significant_5pct']  = tbl['t_ratio'].abs() > CRIT_5
    tbl['significant_1pct']  = tbl['t_ratio'].abs() > CRIT_1
    k   = len(theta)
    aic = -2.0 * ll + 2.0 * k
    bic = -2.0 * ll + k * np.log(T_obs)
    print()
    print('Per-parameter significance (|t| > 1.645 -> 10%, |t| > 1.96 -> 5%, |t| > 2.576 -> 1%):')
    print(tbl.to_string(float_format=lambda x: f'{x: .4f}'))
    print()
    print(f'Log-likelihood : {ll: .4f}')
    print(f'AIC            : {aic: .4f}')
    print(f'BIC            : {bic: .4f}')
    if persist_alpha_beta is not None:
        ab = persist_alpha_beta
        print(f'alpha + beta   : {ab: .4f}   '
              f'({"stationary (<1)" if ab < 1 else "non-stationary (>=1)"})')
    if extra_notes:
        for line in extra_notes:
            print(line)
    print()
    print('Verdict:')
    for name, row in tbl.iterrows():
        print(f'  {name:<12}  t = {row["t_ratio"]:+.2f}   '
              f'p = {row["p_value"]:.4f}   -> {_flag(row["t_ratio"])}')
    return tbl


# === GARCH-X (tone)  --  G&P Eq. (4) with x_{1,t} = Tone_t ================
# sigma^2_t = omega + alpha r^2_{t-1} + beta sigma^2_{t-1} + gamma tone^2_{t-1}
print('==== GARCH-X (tone)  --  G&P Eq. (4) with x = Tone ====')

r_pct    = df['r'] * 100.0
tone = df['tone_mean_x100_winsor']
aligned  = pd.concat([r_pct, tone], axis=1, keys=['r', 'x']).dropna()
r_arr  = aligned['r'].values
x_arr  = aligned['x'].values
T_obs  = len(r_arr)
print(f'Sample after alignment: T = {T_obs} obs '
      f'({aligned.index.min().date()} -> {aligned.index.max().date()})')

PARAM_NAMES = ['omega', 'alpha', 'beta', 'gamma']

def per_obs_ll(params, r, x):
    omega, alpha, beta, gamma = params
    n = len(r)
    sig2 = np.empty(n)
    sig2[0] = max(np.var(r), 1e-8)
    for t in range(1, n):
        s = omega + alpha * r[t-1]**2 + beta * sig2[t-1] + gamma * x[t-1]**2
        sig2[t] = s if s > 1e-8 else 1e-8
    ll_t = -0.5 * (np.log(2.0 * np.pi) + np.log(sig2) + r**2 / sig2)
    return ll_t, sig2

def neg_ll(params, r, x):
    ll_t, _ = per_obs_ll(params, r, x)
    return -ll_t.sum()

x0     = np.array([0.05, 0.10, 0.85, 1e-7])
bounds = [(1e-8, None), (0.0, 0.999), (0.0, 0.999), (None, None)]
opt = minimize(neg_ll, x0, args=(r_arr, x_arr),
               method='L-BFGS-B', bounds=bounds, options={'maxiter': 5000})
if not opt.success:
    print(f'WARNING: optimizer message: {opt.message}')

theta = opt.x
ll    = -opt.fun
se    = _sandwich_se(theta, neg_ll, per_obs_ll, (r_arr, x_arr))

# Bug-6: boundary-solution check
on_bound = []
for _nm, _v, (_lo, _hi) in zip(PARAM_NAMES, theta, bounds):
    if _lo is not None and abs(_v - _lo) < 1e-6: on_bound.append((_nm, _v, 'lower', _lo))
    if _hi is not None and abs(_v - _hi) < 1e-6: on_bound.append((_nm, _v, 'upper', _hi))
if on_bound:
    print('WARNING: boundary solution — SE/t/p not interpretable for:', on_bound)
_report(PARAM_NAMES, theta, se, ll, T_obs,
        persist_alpha_beta=theta[1] + theta[2])

# 3. GARCH-X (article growth) - volume growth of articles matters?

In [6]:
%%capture cap_garchx_art

# === GARCH-X (article growth)  --  G&P Eq. (4) with x_{2,t} = Art_growth_t ==
# sigma^2_t = omega + alpha r^2_{t-1} + beta sigma^2_{t-1} + gamma art_growth^2_{t-1}
print('==== GARCH-X (article growth)  --  G&P Eq. (4) with x = Art_growth ====')

r_pct    = df['r'] * 100.0
artg = df['art_growth_winsor']
aligned  = pd.concat([r_pct, artg], axis=1, keys=['r', 'x']).dropna()
r_arr  = aligned['r'].values
x_arr  = aligned['x'].values
T_obs  = len(r_arr)
print(f'Sample after alignment: T = {T_obs} obs '
      f'({aligned.index.min().date()} -> {aligned.index.max().date()})')

PARAM_NAMES = ['omega', 'alpha', 'beta', 'gamma']

def per_obs_ll(params, r, x):
    omega, alpha, beta, gamma = params
    n = len(r)
    sig2 = np.empty(n)
    sig2[0] = max(np.var(r), 1e-8)
    for t in range(1, n):
        s = omega + alpha * r[t-1]**2 + beta * sig2[t-1] + gamma * x[t-1]**2
        sig2[t] = s if s > 1e-8 else 1e-8
    ll_t = -0.5 * (np.log(2.0 * np.pi) + np.log(sig2) + r**2 / sig2)
    return ll_t, sig2

def neg_ll(params, r, x):
    ll_t, _ = per_obs_ll(params, r, x)
    return -ll_t.sum()

x0     = np.array([0.05, 0.10, 0.85, 0.01])
bounds = [(1e-8, None), (0.0, 0.999), (0.0, 0.999), (None, None)]
opt = minimize(neg_ll, x0, args=(r_arr, x_arr),
               method='L-BFGS-B', bounds=bounds, options={'maxiter': 5000})
if not opt.success:
    print(f'WARNING: optimizer message: {opt.message}')

theta = opt.x
ll    = -opt.fun
se    = _sandwich_se(theta, neg_ll, per_obs_ll, (r_arr, x_arr))

# Bug-6: boundary-solution check
on_bound = []
for _nm, _v, (_lo, _hi) in zip(PARAM_NAMES, theta, bounds):
    if _lo is not None and abs(_v - _lo) < 1e-6: on_bound.append((_nm, _v, 'lower', _lo))
    if _hi is not None and abs(_v - _hi) < 1e-6: on_bound.append((_nm, _v, 'upper', _hi))
if on_bound:
    print('WARNING: boundary solution — SE/t/p not interpretable for:', on_bound)
_report(PARAM_NAMES, theta, se, ll, T_obs,
        persist_alpha_beta=theta[1] + theta[2])

# 4. GARCHAND (tone) - tone sign matters?

In [7]:
%%capture cap_garchand_tone

# === GARCHAND (tone)  --  G&P Eq. (5):  negative tone amplifies the impact ==
# sigma^2_t = omega + alpha r^2_{t-1} + beta sigma^2_{t-1}
#             + gamma (1 + d1 * theta) tone^2_{t-1}
# d1 = 1 if tone_{t-1} < 0, else 0;  theta > 0.
print('==== GARCHAND (tone)  --  G&P Eq. (5) ====')

r_pct    = df['r'] * 100.0
tone = df['tone_mean_x100_winsor']
aligned  = pd.concat([r_pct, tone], axis=1, keys=['r', 'x']).dropna()
r_arr  = aligned['r'].values
x_arr  = aligned['x'].values
T_obs  = len(r_arr)
print(f'Sample after alignment: T = {T_obs} obs '
      f'({aligned.index.min().date()} -> {aligned.index.max().date()})')

PARAM_NAMES = ['omega', 'alpha', 'beta', 'gamma', 'theta']

def per_obs_ll(params, r, x):
    omega, alpha, beta, gamma, theta = params
    n = len(r)
    sig2 = np.empty(n)
    sig2[0] = max(np.var(r), 1e-8)
    for t in range(1, n):
        d1 = 1.0 if x[t-1] < 0 else 0.0
        s = (omega
             + alpha * r[t-1]**2
             + beta  * sig2[t-1]
             + gamma * (1.0 + d1 * theta) * x[t-1]**2)
        sig2[t] = s if s > 1e-8 else 1e-8
    ll_t = -0.5 * (np.log(2.0 * np.pi) + np.log(sig2) + r**2 / sig2)
    return ll_t, sig2

def neg_ll(params, r, x):
    ll_t, _ = per_obs_ll(params, r, x)
    return -ll_t.sum()

x0     = np.array([0.05, 0.10, 0.85, 1e-7, 0.5])
bounds = [(1e-8, None), (0.0, 0.999), (0.0, 0.999), (None, None), (-0.999, None)]
opt = minimize(neg_ll, x0, args=(r_arr, x_arr),
               method='L-BFGS-B', bounds=bounds, options={'maxiter': 5000})
if not opt.success:
    print(f'WARNING: optimizer message: {opt.message}')

theta_hat = opt.x
ll        = -opt.fun
se        = _sandwich_se(theta_hat, neg_ll, per_obs_ll, (r_arr, x_arr))

frac_neg = float((x_arr < 0).mean())
extra = [f'Fraction of days with tone_{{t-1}} < 0 (d1 = 1): {frac_neg:.3f}']

# Bug-6: boundary-solution check
on_bound = []
for _nm, _v, (_lo, _hi) in zip(PARAM_NAMES, theta_hat, bounds):
    if _lo is not None and abs(_v - _lo) < 1e-6: on_bound.append((_nm, _v, 'lower', _lo))
    if _hi is not None and abs(_v - _hi) < 1e-6: on_bound.append((_nm, _v, 'upper', _hi))
if on_bound:
    print('WARNING: boundary solution -- SE/t/p not interpretable for:', on_bound)
_report(PARAM_NAMES, theta_hat, se, ll, T_obs,
        persist_alpha_beta=theta_hat[1] + theta_hat[2],
        extra_notes=extra)

# 5. GARCHAND (article growth) - volume growth of articles sign matters?

In [8]:
%%capture cap_garchand_art

# === GARCHAND (article growth)  --  G&P Eq. (6):  only positive growth bites
# sigma^2_t = omega + alpha r^2_{t-1} + beta sigma^2_{t-1}
#             + gamma d2 art_growth^2_{t-1}
# d2 = 1 if art_growth_{t-1} > 0, else 0;  gamma unrestricted (Bug 5).
print('==== GARCHAND (article growth)  --  G&P Eq. (6) ====')

r_pct    = df['r'] * 100.0
artg = df['art_growth_winsor']           # already on a [-1, ~+6] scale
aligned  = pd.concat([r_pct, artg], axis=1, keys=['r', 'x']).dropna()
r_arr  = aligned['r'].values
x_arr  = aligned['x'].values
T_obs  = len(r_arr)
print(f'Sample after alignment: T = {T_obs} obs '
      f'({aligned.index.min().date()} -> {aligned.index.max().date()})')

PARAM_NAMES = ['omega', 'alpha', 'beta', 'gamma']

def per_obs_ll(params, r, x):
    omega, alpha, beta, gamma = params
    n = len(r)
    sig2 = np.empty(n)
    sig2[0] = max(np.var(r), 1e-8)
    for t in range(1, n):
        d2 = 1.0 if x[t-1] > 0 else 0.0
        s = (omega
             + alpha * r[t-1]**2
             + beta  * sig2[t-1]
             + gamma * d2 * x[t-1]**2)
        sig2[t] = s if s > 1e-8 else 1e-8
    ll_t = -0.5 * (np.log(2.0 * np.pi) + np.log(sig2) + r**2 / sig2)
    return ll_t, sig2

def neg_ll(params, r, x):
    ll_t, _ = per_obs_ll(params, r, x)
    return -ll_t.sum()

x0     = np.array([0.05, 0.10, 0.85, 0.01])
bounds = [(1e-8, None), (0.0, 0.999), (0.0, 0.999), (None, None)]  # Bug 5: gamma unrestricted (relaxed from 1e-8)
opt = minimize(neg_ll, x0, args=(r_arr, x_arr),
               method='L-BFGS-B', bounds=bounds, options={'maxiter': 5000})
if not opt.success:
    print(f'WARNING: optimizer message: {opt.message}')

theta = opt.x
ll    = -opt.fun
se    = _sandwich_se(theta, neg_ll, per_obs_ll, (r_arr, x_arr))

frac_pos = float((x_arr > 0).mean())
extra = [f'Fraction of days with art_growth_{{t-1}} > 0 (d2 = 1): {frac_pos:.3f}']

# Bug-6: boundary-solution check
on_bound = []
for _nm, _v, (_lo, _hi) in zip(PARAM_NAMES, theta, bounds):
    if _lo is not None and abs(_v - _lo) < 1e-6: on_bound.append((_nm, _v, 'lower', _lo))
    if _hi is not None and abs(_v - _hi) < 1e-6: on_bound.append((_nm, _v, 'upper', _hi))
if on_bound:
    print('WARNING: boundary solution — SE/t/p not interpretable for:', on_bound)
_report(PARAM_NAMES, theta, se, ll, T_obs,
        persist_alpha_beta=theta[1] + theta[2],
        extra_notes=extra)

# 6. GARCHND (tone, article growth) - volatility regime matters?

In [9]:
%%capture cap_garchnd

# === GARCHND (tone, article growth)  --  G&P Eq. (7) ========================
# News dummy activated only when previous conditional variance exceeds kappa.
# G&P calibrate kappa to the volatility regime of the underlying asset:
# 10% and 20% annual for OtC FTSE100, 1% and 2% for the lower-vol CtO series.
# REMX has roughly 40% unconditional vol (annualized), so we calibrate the two
# regime thresholds to 30% and 50% annual to keep the dummy informative.
print('==== GARCHND (tone, article growth)  --  G&P Eq. (7) ====')

KAPPA_LOW_ANN  = 30.0
KAPPA_HIGH_ANN = 50.0
def annual_vol_to_daily_var(v_ann_pct):
    return (v_ann_pct / np.sqrt(252.0)) ** 2
KAPPA_LOW  = annual_vol_to_daily_var(KAPPA_LOW_ANN)
KAPPA_HIGH = annual_vol_to_daily_var(KAPPA_HIGH_ANN)
print(f'kappa_low  (vol = {KAPPA_LOW_ANN:>4.1f}% annual) = {KAPPA_LOW:.4f}  (in %^2/day)')
print(f'kappa_high (vol = {KAPPA_HIGH_ANN:>4.1f}% annual) = {KAPPA_HIGH:.4f}  (in %^2/day)')
print()

PARAM_NAMES = ['omega', 'alpha', 'beta', 'gamma']

# Smooth indicator (steep sigmoid) replaces the hard d3 = 1{sigma^2 >= kappa}
# so the log-likelihood becomes differentiable.  With STEEPNESS = 20 the
# transition width is approx. 0.2 in sigma^2 units, much smaller than the
# typical inter-day variation, so the bias relative to G&P's hard dummy is
# negligible while finite-difference scores stay stable.
SMOOTH_K = 20.0
def d3_smooth(z):
    return 1.0 / (1.0 + np.exp(-SMOOTH_K * z))

def per_obs_ll(params, r, x, kappa):
    omega, alpha, beta, gamma = params
    n = len(r)
    sig2 = np.empty(n)
    sig2[0] = max(np.var(r), 1e-8)
    for t in range(1, n):
        d3 = d3_smooth(sig2[t-1] - kappa)
        s = (omega
             + alpha * r[t-1]**2
             + beta  * sig2[t-1]
             + gamma * d3 * x[t-1]**2)
        sig2[t] = s if s > 1e-8 else 1e-8
    ll_t = -0.5 * (np.log(2.0 * np.pi) + np.log(sig2) + r**2 / sig2)
    return ll_t, sig2

def neg_ll(params, r, x, kappa):
    ll_t, _ = per_obs_ll(params, r, x, kappa)
    return -ll_t.sum()

def _fit_robust(neg_ll_fn, x0, bounds, args):
    # Primary: L-BFGS-B. Fallback: Nelder-Mead from the primary result, since
    # the indicator-based recursion can confuse gradient methods.
    opt = minimize(neg_ll_fn, x0, args=args, method='L-BFGS-B',
                   bounds=bounds, options={'maxiter': 5000})
    opt_nm = minimize(neg_ll_fn, opt.x, args=args, method='Nelder-Mead',
                      options={'maxiter': 10000, 'xatol': 1e-7, 'fatol': 1e-7})
    if opt_nm.fun < opt.fun:
        # Project back into bounds and re-run L-BFGS-B for a smooth final step.
        x_proj = np.array([
            min(max(opt_nm.x[i], bounds[i][0] if bounds[i][0] is not None else -np.inf),
                bounds[i][1] if bounds[i][1] is not None else  np.inf)
            for i in range(len(opt_nm.x))
        ])
        opt2 = minimize(neg_ll_fn, x_proj, args=args, method='L-BFGS-B',
                        bounds=bounds, options={'maxiter': 5000})
        if opt2.fun < opt.fun:
            opt = opt2
    return opt

def fit_garchnd(label, x_series_lag, kappa, gamma_x0=0.01):
    r_pct   = df['r'] * 100.0
    aligned = pd.concat([r_pct, x_series_lag], axis=1, keys=['r', 'x']).dropna()
    r_arr = aligned['r'].values
    x_arr = aligned['x'].values
    T_obs = len(r_arr)
    print(f'---- {label} ----')
    print(f'Sample: T = {T_obs} obs '
          f'({aligned.index.min().date()} -> {aligned.index.max().date()})  '
          f'kappa = {kappa:.4f}')

    x0     = np.array([0.086, 0.089, 0.898, gamma_x0])
    bounds = [(1e-8, None), (0.0, 0.999), (0.0, 0.999), (None, None)]
    opt = _fit_robust(neg_ll, x0, bounds, args=(r_arr, x_arr, kappa))
    theta = opt.x
    ll    = -opt.fun

    se = _sandwich_se(theta, neg_ll, per_obs_ll, (r_arr, x_arr, kappa))

    _, sig2_path = per_obs_ll(theta, r_arr, x_arr, kappa)
    frac_active = float((sig2_path[:-1] >= kappa).mean())
    extra = [f'd3 active in-sample (sigma^2_{{t-1}} >= kappa): {frac_active:.3f}']

    _report(PARAM_NAMES, theta, se, ll, T_obs,
            persist_alpha_beta=theta[1] + theta[2],
            extra_notes=extra)
    # Bug-6: boundary-solution check
    on_bound = []
    for _nm, _v, (_lo, _hi) in zip(PARAM_NAMES, theta, bounds):
        if _lo is not None and abs(_v - _lo) < 1e-6: on_bound.append((_nm, _v, 'lower', _lo))
        if _hi is not None and abs(_v - _hi) < 1e-6: on_bound.append((_nm, _v, 'upper', _hi))
    if on_bound:
        print('WARNING: boundary solution -- SE/t/p not interpretable for:', on_bound)
    print()

tone_lag = df['tone_mean_x100_winsor']
artg_lag = df['art_growth_winsor']

fit_garchnd('GARCHND  x = Tone           kappa = 30% annual', tone_lag, KAPPA_LOW,  gamma_x0=1e-7)
fit_garchnd('GARCHND  x = Tone           kappa = 50% annual', tone_lag, KAPPA_HIGH, gamma_x0=1e-7)
fit_garchnd('GARCHND  x = Art_growth     kappa = 30% annual', artg_lag, KAPPA_LOW)
fit_garchnd('GARCHND  x = Art_growth     kappa = 50% annual', artg_lag, KAPPA_HIGH)

# Generate report

In [10]:
from pathlib import Path

REPORTS_DIR = Path("REPORTS")
REPORTS_DIR.mkdir(exist_ok=True)

sections = [
    ("GARCH(1,1) -- Gaussian baseline",         cap_garch),
    ("GARCH-X (tone)",                          cap_garchx_tone),
    ("GARCH-X (article growth)",                cap_garchx_art),
    ("GARCHAND (tone)",                         cap_garchand_tone),
    ("GARCHAND (article growth)",               cap_garchand_art),
    ("GARCHND (tone, art_growth)",              cap_garchnd),
]

def _banner(title, char='='):
    bar = char * 78
    return f"{bar}\n{title}\n{bar}\n"

out_lines = []
for title, cap in sections:
    out_lines.append(_banner(title, '-'))
    out_lines.append(cap.stdout if cap.stdout else '(no stdout captured)\n')
    out_lines.append("\n")
combined = "".join(out_lines)

(REPORTS_DIR / "03-GP-models-gaussian.txt").write_text(combined)
print(combined)
print(f"Persisted combined GARCH-family capture -> REPORTS/03-GP-models-gaussian.txt")


------------------------------------------------------------------------------
GARCH(1,1) -- Gaussian baseline
------------------------------------------------------------------------------
                       Zero Mean - GARCH Model Results                        
Dep. Variable:                      r   R-squared:                       0.000
Mean Model:                 Zero Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -5982.31
Distribution:                  Normal   AIC:                           11970.6
Method:            Maximum Likelihood   BIC:                           11988.4
                                        No. Observations:                 2765
Date:                Fri, May 29 2026   Df Residuals:                     2765
Time:                        00:17:09   Df Model:                            0
                             Volatility Model                             
                 coef   

## Old-vs-new: sign and significance of γ and θ (Gaussian innovations)

Reference (OLD): last report in `REPORTS/03-GP-models-gaussian.txt` (before any bug fixes).
Fixes applied: Bug 1 (GARCHND double-lag removed), Bug 3 (tone×100 + winsorization),
Bug 4 (zero-mean returns), Bug 5 (θ/γ bounds relaxed in GARCHAND).
NEW values: re-run this notebook top-to-bottom and read from the cells above.

| Model | Param | OLD sign | OLD signif. | NEW sign | NEW signif. | Notes |
|---|---|---|---|---|---|---|
| GARCH-X tone | γ | + | n.s. | TBD | TBD | Bug 3: tone×100 winsorized |
| GARCH-X art | γ | − | 5% | TBD | TBD | Bug 3: art_growth winsorized |
| GARCHAND tone | γ | − | n.s. | TBD | TBD | Bug 3+5: tone×100 winsorized; bounds unchanged |
| GARCHAND tone | θ | + | n.s. | TBD | TBD | Bug 5: bound relaxed to (−0.999, None); old run was boundary solution |
| GARCHAND art | γ | +0.00 | n.s. | TBD | TBD | Bug 5: bound relaxed to (None, None); old γ was pinned at 0 |
| GARCHND tone κ=30% | γ | + | n.s. | TBD | TBD | Bug 1: lag corrected; Bug 3: col ref |
| GARCHND tone κ=50% | γ | − | 1% | TBD | TBD | Bug 1: lag corrected; Bug 3: col ref |
| GARCHND art κ=30% | γ | + | n.s. | TBD | TBD | Bug 1: lag corrected; Bug 3: col ref |
| GARCHND art κ=50% | γ | − | n.s. | TBD | TBD | Bug 1: lag corrected; Bug 3: col ref |

**Interpretation note (deferred)**: Do not read into sign flips here — the point of this table is
to record what the data say after the implementation bugs are corrected, not to interpret results.